# Smart Pointers

Welcome! Smart pointers are data structures that act like pointers but add extra capabilities. In Rust, they're implemented as structs that implement the `Deref` and `Drop` traits. Unlike regular references (`&`), smart pointers own the data they point to. This guide covers `Box`, `Rc`, `Arc`, `RefCell`, `Weak`, and `Cow` — and when to use each.

## What Are Smart Pointers?

A **smart pointer** is a pointer-like type that manages a resource (usually heap-allocated memory) and provides additional behavior. In Rust:

- **References** (`&T`) borrow data; they don't own it.
- **Smart pointers** own data and may allow shared or mutable access with extra guarantees.

**Analogy:** A reference is like looking at a book on someone's shelf. A smart pointer is like having your own copy in a box — you own it, and the "box" might add features (like reference counting or interior mutability).

## Box<T>: Heap Allocation

`Box<T>` stores data on the heap. Use it when:
- You have a type whose size isn't known at compile time (e.g., recursive types)
- You want to transfer ownership of large data without copying
- You need to store a trait object (dyn Trait)

In [ ]:
// Simple heap allocation
let b = Box::new(42);
println!("Box holds: {}", b);

// Box dereferences automatically
let x = *b;
println!("Dereferenced: {}", x);

## Box for Recursive Types

Rust needs to know the size of every type at compile time. A recursive type like a linked list would be infinitely large — unless we put the "next" node in a `Box`, which has a fixed size (pointer).

In [ ]:
// Without Box, this would be infinite size!
fn cons_list_demo() {
    enum List {
        Cons(i32, Box<List>),
        Nil,
    }
    let list = List::Cons(1, Box::new(List::Cons(2, Box::new(List::Cons(3, Box::new(List::Nil))))));
    // In a real impl you'd traverse it; here we just show it compiles
}
cons_list_demo();
println!("Recursive List type works!");

## Deref and DerefMut Traits

The `Deref` trait lets you use `*` to get the inner value. **Deref coercion** automatically converts `&Box<T>` to `&T` when passing to functions — so you can pass a `&Box<String>` where `&str` is expected.

In [ ]:
fn print_len(s: &str) {
    println!("Length: {}", s.len());
}

let boxed = Box::new(String::from("hello"));
// Deref coercion: &Box<String> -> &String -> &str
print_len(&boxed);

## Drop Trait

The `Drop` trait lets you run code when a value goes out of scope. `Box` implements `Drop` to free the heap memory. You can implement `Drop` for your own types to clean up resources.

In [ ]:
struct Droppable(i32);

impl Drop for Droppable {
    fn drop(&mut self) {
        println!("Dropping value {}", self.0);
    }
}

{
    let d = Droppable(42);
    println!("Inside scope");
}
println!("Left scope - Drop was called");

## Rc<T>: Reference Counting (Shared Ownership)

`Rc<T>` (Reference Counted) lets multiple owners share the same data. The data is dropped when the last `Rc` is dropped. Use it for **single-threaded** shared ownership. It's not thread-safe.

In [ ]:
use std::rc::Rc;

let shared = Rc::new(String::from("shared data"));
let clone1 = Rc::clone(&shared);
let clone2 = Rc::clone(&shared);

println!("Strong count: {}", Rc::strong_count(&shared));
println!("All point to: {}", shared);

## Arc<T>: Atomic Reference Counting (Thread-Safe)

`Arc<T>` is like `Rc` but uses atomic operations, so it's safe to share across threads. Use `Arc` when you need shared ownership in concurrent code.

In [ ]:
use std::sync::Arc;

let shared = Arc::new(42);
let clone1 = Arc::clone(&shared);
let clone2 = Arc::clone(&shared);

println!("Arc value: {}", shared);
// In real code you'd pass clone1/clone2 to threads

## RefCell<T>: Interior Mutability

`RefCell<T>` allows mutable borrows checked at **runtime** instead of compile time. You can have an immutable reference to a `RefCell` but still mutate the inner value via `borrow_mut()`. Use when you need mutability through an immutable interface.

In [ ]:
use std::cell::RefCell;

let data = RefCell::new(0);
{
    let mut borrow = data.borrow_mut();
    *borrow += 1;
}
println!("Value: {}", data.borrow());

## Rc<RefCell<T>>: Shared Mutable Data

Combine `Rc` and `RefCell` when you need **multiple owners** that can **mutate** the data. `Rc` gives shared ownership; `RefCell` gives interior mutability.

In [ ]:
use std::rc::Rc;
use std::cell::RefCell;

let shared_mut = Rc::new(RefCell::new(0));
let c1 = Rc::clone(&shared_mut);
let c2 = Rc::clone(&shared_mut);

*c1.borrow_mut() += 10;
*c2.borrow_mut() += 5;
println!("Final value: {}", *shared_mut.borrow());

## Weak<T>: Breaking Reference Cycles

`Rc` can create cycles (A points to B, B points to A) so nothing is ever dropped. `Weak<T>` is a non-owning reference that doesn't keep the data alive. Use it for back-references in graphs or trees.

In [ ]:
use std::rc::{Rc, Weak};

let strong = Rc::new(42);
let weak: Weak<i32> = Rc::downgrade(&strong);

// Upgrade gives Option<Rc<T>>
if let Some(rc) = weak.upgrade() {
    println!("Weak upgraded: {}", rc);
}

drop(strong);
assert!(weak.upgrade().is_none());
println!("After dropping strong, weak.upgrade() is None");

## Cow<T>: Clone-on-Write

`Cow` (Clone on Write) can hold either borrowed or owned data. When you need to mutate, it clones the data. Useful for avoiding unnecessary allocations when you might not need to modify.

In [ ]:
use std::borrow::Cow;

fn add_prefix(s: &str) -> Cow<str> {
    if s.starts_with("Hello") {
        Cow::Borrowed(s)
    } else {
        Cow::Owned(format!("Hello, {}", s))
    }
}

let s1 = "Hello world";
let s2 = "Rust";
println!("{}", add_prefix(s1));
println!("{}", add_prefix(s2));

## When to Use Which Smart Pointer

| Need | Use |
|------|-----|
| Heap allocation, recursive types, trait objects | `Box<T>` |
| Single-threaded shared ownership | `Rc<T>` |
| Thread-safe shared ownership | `Arc<T>` |
| Mutate through immutable reference (single-threaded) | `RefCell<T>` |
| Shared + mutable (single-threaded) | `Rc<RefCell<T>>` |
| Break reference cycles | `Weak<T>` |
| Avoid cloning when maybe not needed | `Cow<T>` |

## Common Mistakes Beginners Make

1. **Using `Rc` when `Box` would do** — If you have single ownership, prefer `Box`.
2. **Using `Rc` across threads** — Use `Arc` for concurrency.
3. **Panicking with RefCell** — `borrow_mut()` panics if already borrowed; use `try_borrow_mut()` for fallible access.
4. **Creating cycles with Rc** — Use `Weak` for back-references.
5. **Overusing RefCell** — Prefer the borrow checker when possible; RefCell moves checks to runtime.

## Key Rules to Remember

- `Box<T>` owns heap data; use for recursion, large data, trait objects.
- `Deref` enables `*` and deref coercion; `Drop` runs cleanup on scope exit.
- `Rc` = shared ownership, single-threaded; `Arc` = shared ownership, thread-safe.
- `RefCell` = interior mutability with runtime borrow checking.
- `Rc<RefCell<T>>` = shared mutable data in single-threaded code.
- `Weak` breaks cycles; `Cow` avoids unnecessary clones.